# Capitulo 02 - Diagnostico y priorizacion

Este notebook muestra una priorizacion reproducible con territorios ficticios. Los datos son simulados y no deben usarse como evidencia real.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_book_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        nested = base / "books" / "politicas_publicas_basadas_en_datos"
        if nested.exists():
            return nested
        if base.name == "politicas_publicas_basadas_en_datos" and (base / "PROJECT_BRIEF.md").exists():
            return base
    raise FileNotFoundError("No se encontro la raiz del libro.")


BOOK_ROOT = find_book_root()
TABLES_DIR = BOOK_ROOT / "outputs" / "tables"
FIGURES_DIR = BOOK_ROOT / "outputs" / "figures"
REPORTS_DIR = BOOK_ROOT / "outputs" / "reports"

for path in [TABLES_DIR, FIGURES_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

BOOK_ROOT

In [ ]:
datos = pd.DataFrame(
    {
        "territorio": ["Zona A", "Zona B", "Zona C", "Zona D", "Zona E", "Zona F"],
        "necesidad": [82, 67, 91, 54, 73, 60],
        "factibilidad": [55, 80, 40, 72, 62, 88],
        "equidad": [76, 58, 94, 45, 81, 50],
    }
)

datos

In [ ]:
def minmax(serie: pd.Series) -> pd.Series:
    return (serie - serie.min()) / (serie.max() - serie.min())


pesos = {"necesidad": 0.45, "factibilidad": 0.25, "equidad": 0.30}

ranking = datos.copy()
for columna in pesos:
    ranking[f"{columna}_norm"] = minmax(ranking[columna])

ranking["puntaje_prioridad"] = sum(
    pesos[columna] * ranking[f"{columna}_norm"] for columna in pesos
)

ranking = ranking.sort_values("puntaje_prioridad", ascending=False).reset_index(drop=True)
ranking["posicion"] = ranking.index + 1
ranking

In [ ]:
tabla_salida = TABLES_DIR / "chapter_02_ranking_priorizacion.csv"
ranking.to_csv(tabla_salida, index=False, encoding="utf-8")

fig, ax = plt.subplots(figsize=(8, 4.8))
plot_data = ranking.sort_values("puntaje_prioridad")
ax.barh(plot_data["territorio"], plot_data["puntaje_prioridad"], color="#2f6f73")
ax.set_title("Ranking pedagogico de prioridad")
ax.set_xlabel("Puntaje normalizado")
ax.set_ylabel("Territorio ficticio")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()

figura_salida = FIGURES_DIR / "chapter_02_priorizacion.png"
fig.savefig(figura_salida, dpi=180)

reporte_salida = REPORTS_DIR / "chapter_02_nota_metodologica.md"
reporte_salida.write_text(
    "# Nota metodologica - Capitulo 02\n\n"
    "Los territorios y valores son ficticios. El ranking ensena normalizacion, "
    "ponderacion y sensibilidad, pero no recomienda una asignacion real de "
    "presupuesto. La priorizacion no estima impacto causal.\n\n"
    f"Pesos usados: {pesos}.\n",
    encoding="utf-8",
)

tabla_salida, figura_salida, reporte_salida

## Interpretacion

El puntaje combina necesidad, factibilidad y equidad con pesos explicitos. Cambiar los pesos puede cambiar el ranking; por eso la priorizacion debe acompaniarse de analisis de sensibilidad y deliberacion institucional.